# 02 — Analyse rule sensitivity and card effectiveness

This notebook analyses the saved rule-sensitivity simulations generated by `01_run_rule_sensitivity.ipynb`.

The analysis addresses two related questions:

1. **Rule sensitivity:** how does the size and structure of the choice set affect cautious versus greedy players?
2. **Card robustness:** which cards remain associated with good outcomes across different card-access and passing rules?

Victory points (`vp`) represent the game's CO₂ score. **Lower VP is better.**


## 1. Load the experiment

The notebook uses the combined Parquet files created by the run notebook. The individual condition folders remain available for debugging or rerunning specific conditions.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from model.game import load_game_data
from model.analysis import card_analysis


VERSION = Path("Versionen/paper_draft_v1")
EXPERIMENT_DIR = VERSION / "results" / "rule_sensitivity"

game_data = load_game_data(VERSION)

all_games = pd.read_parquet(
    EXPERIMENT_DIR / "all_games.parquet"
)

all_plays = pd.read_parquet(
    EXPERIMENT_DIR / "all_plays.parquet"
)

design = pd.read_csv(
    EXPERIMENT_DIR / "design.csv"
)

print(f"{len(all_games):,} games")
print(f"{len(all_plays):,} logged decisions")
print(f"{all_games['experiment'].nunique()} conditions")


## 2. Overall outcome summary

First inspect whether the gameplay rules materially change the game.

The summary includes:

- mean and median final VP,
- dispersion of VP,
- remaining budget,
- number of cards played,
- share of games completing all four rounds.

The number of cards played is particularly important because passing probability directly affects how aggressively the simulated player continues selecting cards.


In [ ]:
summary = (
    all_games
    .groupby(["access_rule", "pass_probability"])
    .agg(
        games=("game_id", "size"),
        mean_vp=("vp", "mean"),
        median_vp=("vp", "median"),
        std_vp=("vp", "std"),
        mean_budget=("budget", "mean"),
        mean_cards=("n_cards_played", "mean"),
        completed_share=(
            "rounds_played",
            lambda x: (x == 4).mean(),
        ),
    )
    .reset_index()
)

summary.to_csv(
    EXPERIMENT_DIR / "summary.csv",
    index=False,
)

summary


## 3. Choice set × passing behaviour

The following pivot tables are the core rule-sensitivity results.

Read each row as one player type:

- `0.00`: never voluntarily passes; most greedy
- `0.10`: low probability of stopping
- `0.25`: intermediate
- `0.50`: frequently stops; most cautious

The columns vary what cards are available to choose from.


In [ ]:
vp_table = all_games.pivot_table(
    index="pass_probability",
    columns="access_rule",
    values="vp",
    aggfunc="mean",
)

vp_table


In [ ]:
cards_played_table = all_games.pivot_table(
    index="pass_probability",
    columns="access_rule",
    values="n_cards_played",
    aggfunc="mean",
)

cards_played_table


### Mean final VP

A lower line means better average game performance.

The comparison within each pass probability answers the main question: **how much does broader or differently structured card access matter for players with the same stopping behaviour?**


In [ ]:
ax = vp_table.plot(
    marker="o",
    figsize=(8, 5),
)

ax.set_xlabel("Pass probability")
ax.set_ylabel("Mean final VP")
ax.set_title("Game performance by choice set and passing behaviour")

plt.show()


### Mean number of cards played

This plot helps interpret the VP result. If one rule produces better VP primarily because it causes players to select many more cards, that should be visible here.


In [ ]:
ax = cards_played_table.plot(
    marker="o",
    figsize=(8, 5),
)

ax.set_xlabel("Pass probability")
ax.set_ylabel("Mean cards played")
ax.set_title("Cards played by choice set and passing behaviour")

plt.show()


## 4. Card effectiveness within each condition

Card effectiveness should **not** be estimated from all 200,000 games pooled together.

The access rules directly change the probability that a card is offered and playable. `card_analysis()` is therefore calculated separately for each of the 20 experimental conditions.

The main card metrics are:

- `occurrence`: fraction of games containing the card,
- `elite_occurrence`: fraction of the best 5% of games containing it,
- `enrichment`: elite occurrence divided by overall occurrence,
- `vp_lift`: mean VP without the card minus mean VP with it; **positive is better**,
- `selection_rate`: selection frequency conditional on being playable.

These remain descriptive associations rather than causal card effects.


In [ ]:
card_results = []

for experiment, games in all_games.groupby("experiment"):

    plays = all_plays.loc[
        all_plays["experiment"] == experiment
    ]

    result = card_analysis(
        games,
        plays=plays,
        game_data=game_data,
        elite_share=0.05,
    )

    result["experiment"] = experiment
    result["access_rule"] = games["access_rule"].iloc[0]
    result["pass_probability"] = games["pass_probability"].iloc[0]

    card_results.append(result)

card_results = pd.concat(
    card_results,
    ignore_index=True,
)

card_results.to_parquet(
    EXPERIMENT_DIR / "card_results.parquet",
    index=False,
)

card_results.head()


## 5. Robustness of individual cards

A card is more convincing when its association with good outcomes persists across many rule conditions.

For each card, the table therefore summarises:

- average, minimum, and maximum VP lift,
- number of conditions where VP lift can actually be estimated,
- share of estimable conditions with positive VP lift,
- mean elite enrichment,
- share of conditions with enrichment above 1.

`vp_lift` cannot be estimated when a card occurs in every game or in no game under a condition. Those cases are excluded from `positive_lift_share`.


In [ ]:
card_robustness = (
    card_results
    .groupby(["card", "label_en"])
    .agg(
        conditions=("experiment", "nunique"),
        mean_occurrence=("occurrence", "mean"),

        mean_vp_lift=("vp_lift", "mean"),
        min_vp_lift=("vp_lift", "min"),
        max_vp_lift=("vp_lift", "max"),

        lift_estimable=("vp_lift", lambda x: x.notna().sum()),
        positive_lift_cases=("vp_lift", lambda x: (x > 0).sum()),

        mean_enrichment=("enrichment", "mean"),
        min_enrichment=("enrichment", "min"),
        max_enrichment=("enrichment", "max"),
        enriched_cases=("enrichment", lambda x: (x > 1).sum()),
    )
    .reset_index()
)

card_robustness["positive_lift_share"] = (
    card_robustness["positive_lift_cases"]
    / card_robustness["lift_estimable"].replace(0, pd.NA)
)

card_robustness["enriched_share"] = (
    card_robustness["enriched_cases"]
    / card_robustness["conditions"]
)

card_robustness = card_robustness.sort_values(
    ["positive_lift_share", "mean_vp_lift"],
    ascending=False,
)

card_robustness.to_csv(
    EXPERIMENT_DIR / "card_robustness.csv",
    index=False,
)

card_robustness


## 6. Strong robust candidates

This is a screening view rather than a formal statistical threshold.

It selects cards whose VP lift is positive in at least 75% of the rule conditions where the lift can be estimated. The resulting cards should then be inspected individually.


In [ ]:
robust_candidates = card_robustness.loc[
    (card_robustness["lift_estimable"] > 0)
    & (card_robustness["positive_lift_share"] >= 0.75)
].copy()

robust_candidates[
    [
        "card",
        "label_en",
        "mean_occurrence",
        "mean_vp_lift",
        "min_vp_lift",
        "max_vp_lift",
        "positive_lift_share",
        "mean_enrichment",
        "enriched_share",
    ]
]


## 7. Inspect one card across all rule conditions

Set `CARD` to any semantic `card_id`.

The two pivot tables show how VP lift and elite enrichment change with both experimental factors. This is useful for identifying cards whose apparent value depends strongly on choice-set size or player caution.


In [ ]:
CARD = card_robustness.iloc[0]["card"]

card_vp_lift = (
    card_results.loc[
        card_results["card"] == CARD
    ]
    .pivot(
        index="pass_probability",
        columns="access_rule",
        values="vp_lift",
    )
)

card_vp_lift


In [ ]:
ax = card_vp_lift.plot(
    marker="o",
    figsize=(8, 5),
)

ax.axhline(0, linestyle="--")
ax.set_xlabel("Pass probability")
ax.set_ylabel("VP lift")
ax.set_title(f"VP lift across rules: {CARD}")

plt.show()


In [ ]:
card_enrichment = (
    card_results.loc[
        card_results["card"] == CARD
    ]
    .pivot(
        index="pass_probability",
        columns="access_rule",
        values="enrichment",
    )
)

card_enrichment


## 8. Interpretation checklist

For the formal investigation, interpret results in this order:

1. **Game-level rule effect:** Does broader or structured card access change final VP?
2. **Behaviour interaction:** Does that effect change with pass probability?
3. **Cards played:** Are performance changes largely explained by how many cards are selected?
4. **Card occurrence:** Is an individual card observed often enough for comparison?
5. **VP lift:** Is the card associated with lower final VP?
6. **Elite enrichment:** Is it disproportionately represented among the best games?
7. **Rule robustness:** Do these associations persist across the 20 experimental conditions?
8. **Opportunity:** When available, is the card actually selected?

A later strategy experiment can test whether these conclusions also survive deliberate technology preferences rather than random card selection.
